# Material + Reaction Rate - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC **transport-coupled** material depletion
using the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Method**: transport-coupled depletion. A Monte Carlo neutron transport solve
  runs at every irradiation step; the flux-weighted one-group reaction rates it
  produces drive the burnup ODE. YAMC uses
  `Model.simulate_transmutation(method="coupled", ...)`; OpenMC uses
  `openmc.deplete.CoupledOperator` with a depletion integrator.
- **Difference from the flux-spectrum sibling**: the
  `material-flux-spectrum` notebook prescribes a fixed multigroup flux and never
  transports particles (`Material.transmute` / `material.deplete(multigroup_flux=...)`).
  Here the reaction rates are *computed from transport* against the actual
  geometry and source each step, so geometry, self-shielding and spectrum are
  modelled rather than assumed.
- **Geometry**: a solid sphere (vacuum boundary) with a 14.06 MeV point neutron
  source at the origin. The cell volume is computed by stochastic sampling.
- **Material**: a single seed nuclide at 1.0 g/cm3, 294 K, marked depletable.
- **Chain**: `chain_endf_b8.1_sfr.xml` for OpenMC; the cached YAMC arrow chain.
  Both reduced per nuclide to `reduce_chain_level=2`.
- **Timesteps**: 3 x 1 hour irradiation + 2 x 15 day cooling.
- **Comparison**: nuclide atom densities at each timestep.

> This notebook only **runs and caches** the simulations. The results are explored interactively in the [benchmark database](/benchmarks.html), which renders the flux-spectrum comparisons, reaction-rate tables and geometry per benchmark. See the book intro for the link.

In [ ]:
# Setup: locate the yamc_verify package (repo root) and load this task.
import sys
from pathlib import Path as _Path
_repo = _Path.cwd().resolve()
while _repo != _repo.parent and not (_repo / "yamc_verify").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from yamc_verify import backends, get_task
from yamc_verify.families import transmutation

TASK = get_task("transmutation/material-reaction-rate/endf-b8.1")
BACKENDS = backends.resolve()
print(f"Backends: {BACKENDS.tokens}")

In [ ]:
# Run all backends for every discovered nuclide (results cached on disk).
all_results, first_yamc_model = transmutation.run_all(TASK, BACKENDS)